# Phase 3 — Dataset Validation

Three-level audit of the train/val/test datasets produced by
`phase3_dataset_construction.ipynb`. Mirrors the pattern in
`phase2_validation.ipynb`.

- **Level 1**: structural integrity — shapes, schemas, disjoint splits, no duplicates
- **Level 2**: label sanity — stratification preserved, per-persona churn rates match the data design
- **Level 3**: feature sanity — NaN audit, leakage check, cross-format consistency, decay signal visible in features

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/processed"
RAW_DIR  = "../data/raw"

# Datasets
xgb = {s: pd.read_parquet(f"{DATA_DIR}/xgboost_{s}.parquet") for s in ["train","val","test"]}
lstm = {s: dict(np.load(f"{DATA_DIR}/lstm_{s}.npz", allow_pickle=True)) for s in ["train","val","test"]}

# Reference data
users = pd.read_parquet(f"{RAW_DIR}/users.parquet", columns=["user_id","persona"])

for s in ["train","val","test"]:
    print(f"{s:<5}  xgb={xgb[s].shape}  lstm X={lstm[s]['X'].shape}  y={lstm[s]['y'].shape}")

## Level 1 — Structural Integrity

In [ ]:
# 1.1 Total rows = sum of split rows for both formats
n_total_xgb = sum(len(xgb[s]) for s in xgb)
n_total_lstm = sum(lstm[s]["X"].shape[0] for s in lstm)
assert n_total_xgb == 50000, f"XGBoost total: {n_total_xgb}"
assert n_total_lstm == 50000, f"LSTM total: {n_total_lstm}"
print(f"PASS: 50,000 users covered by both XGBoost and LSTM splits")

In [ ]:
# 1.2 Splits are disjoint and exhaustive
all_xgb_ids = set()
for s in xgb:
    ids = set(xgb[s].index)
    assert len(all_xgb_ids & ids) == 0, f"{s} overlaps another XGB split"
    all_xgb_ids |= ids
assert all_xgb_ids == set(users["user_id"]), "XGB splits do not cover all users"

all_lstm_ids = set()
for s in lstm:
    ids = set(lstm[s]["user_ids"])
    assert len(all_lstm_ids & ids) == 0, f"{s} overlaps another LSTM split"
    all_lstm_ids |= ids
assert all_lstm_ids == set(users["user_id"]), "LSTM splits do not cover all users"
print("PASS: XGBoost and LSTM splits each exhaustive and disjoint")

In [ ]:
# 1.3 Cross-format consistency — same user_id partition across XGB and LSTM
for s in ["train","val","test"]:
    xgb_ids = set(xgb[s].index)
    lstm_ids = set(lstm[s]["user_ids"])
    only_xgb = xgb_ids - lstm_ids
    only_lstm = lstm_ids - xgb_ids
    assert only_xgb == set(), f"{s}: in XGB but not LSTM: {len(only_xgb)}"
    assert only_lstm == set(), f"{s}: in LSTM but not XGB: {len(only_lstm)}"
print("PASS: every user_id appears in the same split for both formats")

In [ ]:
# 1.4 Schema consistency across splits
xgb_cols = [list(xgb[s].columns) for s in ["train","val","test"]]
assert xgb_cols[0] == xgb_cols[1] == xgb_cols[2], "XGB columns differ across splits"
for s in ["train","val","test"]:
    assert lstm[s]["X"].shape[1:] == (4, 38), f"{s} LSTM shape: {lstm[s]['X'].shape}"
    assert list(lstm[s]["feature_names"]) == list(lstm["train"]["feature_names"]), \
        f"{s} LSTM feature names differ"
print(f"PASS: schemas consistent (XGB {len(xgb_cols[0])} cols, LSTM 38 feats × 4 weeks)")

## Level 2 — Label Sanity

In [ ]:
# 2.1 Stratification — each split has the same churn rate (~12.2%)
print(f"{'split':<6} {'users':>7} {'churn':>7} {'rate':>7}")
for s in ["train","val","test"]:
    n = len(xgb[s])
    cn = int(xgb[s]["churn"].sum())
    rate = cn / n
    print(f"{s:<6} {n:>7,} {cn:>7,} {rate:>7.1%}")

rates = [xgb[s]["churn"].mean() for s in ["train","val","test"]]
assert max(rates) - min(rates) < 0.005, f"Split churn rates differ too much: {rates}"
print("\nPASS: stratification preserved (rates within 0.5 pp)")

In [ ]:
# 2.2 LSTM y matches XGBoost churn label
for s in ["train","val","test"]:
    xgb_y = xgb[s].loc[lstm[s]["user_ids"], "churn"].to_numpy()
    lstm_y = lstm[s]["y"]
    assert np.array_equal(xgb_y, lstm_y), f"{s}: XGB churn != LSTM y"
print("PASS: LSTM y arrays match XGBoost churn column")

In [ ]:
# 2.3 Per-persona churn rate matches the data-generation design
labels_all = pd.concat(
    [xgb[s][["churn"]].assign(split=s) for s in ["train","val","test"]]
)
labels_all = labels_all.merge(users, left_index=True, right_on="user_id")
persona_stats = labels_all.groupby("persona")["churn"].agg(["sum","count","mean"]).round(4)
print(persona_stats)
print()

# Expected from data design: about_to_churn dominates; hardcore/regular should be near zero.
assert persona_stats.loc["about_to_churn","mean"] > 0.5, "about_to_churn churn rate too low"
assert persona_stats.loc["hardcore","mean"]       < 0.01, "hardcore should rarely churn"
assert persona_stats.loc["regular","mean"]        < 0.05, "regular should rarely churn"
print("PASS: persona churn rates match data design")

## Level 3 — Feature Sanity

In [ ]:
# 3.1 Leakage check — `persona` MUST NOT be a feature column
forbidden = {"persona","subscription_tier","signup_date","user_id"}
xgb_features = set(xgb["train"].columns) - {"churn"}
lstm_features = set(lstm["train"]["feature_names"])

xgb_leak  = xgb_features  & forbidden
lstm_leak = lstm_features & forbidden
assert xgb_leak  == set(), f"XGB leak: {xgb_leak}"
assert lstm_leak == set(), f"LSTM leak: {lstm_leak}"
print(f"PASS: no leakage. XGB features={len(xgb_features)}, LSTM features={len(lstm_features)}")

In [ ]:
# 3.2 LSTM tensor has no NaN (asserted at build time, re-check here)
for s in ["train","val","test"]:
    assert not np.isnan(lstm[s]["X"]).any(), f"{s} LSTM has NaN"
print("PASS: LSTM tensors are NaN-free across all splits")

In [ ]:
# 3.3 XGBoost NaN inventory (XGBoost handles these natively — just audit)
nulls = xgb["train"].isnull().sum()
print("XGBoost columns with NaN (train split):")
print(nulls[nulls > 0].sort_values(ascending=False).to_string())

In [ ]:
# 3.4 Decay signal visible: churners should have much lower weekly_session_count_last
#     and steeper drop in playtime than non-churners.
for s in ["train"]:
    grouped = xgb[s].groupby("churn")[[
        "weekly_session_count","weekly_session_count_last",
        "total_playtime_min","total_playtime_min_last",
        "playtime_slope","longest_inactive_days_last",
    ]].mean().round(2)
    print(grouped.T)

train_mean = xgb["train"].groupby("churn")[["weekly_session_count_last"]].mean()
assert train_mean.loc[1,"weekly_session_count_last"] < train_mean.loc[0,"weekly_session_count_last"] * 0.5, \
    "Churners should have <50% of non-churners' last-week sessions"
print("\nPASS: churners show <50% last-week sessions vs non-churners (decay learnable)")

## Summary

In [ ]:
print("=" * 55)
print("Phase 3 Dataset Validation")
print("=" * 55)
print(f"Total users:     {sum(len(xgb[s]) for s in xgb):,}")
for s in ["train","val","test"]:
    print(f"  {s:<5}  {len(xgb[s]):>6,}  churn={xgb[s]['churn'].mean():.1%}")
print()
print(f"XGB feature cols: {len(xgb['train'].columns) - 1}")
print(f"LSTM features:    {len(lstm['train']['feature_names'])}")
print(f"LSTM tensor:      (N, 4, 38) per split, NaN-free")
print()
print("All structural, label, and feature checks PASSED.")
print("=" * 55)